# IMPORTS

In [2]:
import pickle as pkl
from torch import nn
import os
import random
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
import torch.nn.functional as F
import plotly.graph_objects as go
from matplotlib import cm
import wandb

# PARAMS

In [3]:
# PARAMS
max_length = 50

parent_folder = "nanos_networkx_small"  # Update this to your data path - this is relative!
chunk_length = 50
max_proteins = 3000  # Limit number of proteins for faster execution
batch_size = 32
lr = 1e-4 # learning rate
num_epochs = 10
min_epochs = 1
patience = 30

# More aggressive subsequence parameters
min_subseq_length = 20  # Even smaller minimum subsequence length
step_size = 1  # Much smaller step size for more overlap
subgraph_limit = max_proteins * 300 # max number of subgraphs


model_path = "dual_output_gran_model.pt"  # Path for saving/loading model


# Set device
def get_device():
    """
    Returns the best available device for PyTorch operations.
    Priority: CUDA GPU > MPS (Apple Silicon) > CPU
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print(f"Using device: {device}")




Using device: mps


# MODEL CLASSES - to be split out

In [4]:
#split out
class GraphAttention(nn.Module):
    def __init__(self, in_features, out_features, n_heads=4, dropout=0.1, alpha=0.2):
        super(GraphAttention, self).__init__()
        self.in_features = in_features
        self.out_features = out_features  # out_features_per_head
        self.n_heads = n_heads
        self.dropout = dropout
        self.alpha = alpha

        # Linear transformations
        self.W = nn.Linear(in_features, n_heads * out_features, bias=False)
        self.a = nn.Linear(2 * out_features, 1, bias=False)

        # Initialize parameters
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a.weight)

        self.leakyrelu = nn.LeakyReLU(self.alpha)
        self.dropout_layer = nn.Dropout(self.dropout)

    def forward(self, h, adj):
        # h: [batch_size, num_nodes, in_features] (32, 50, 128)
        # adj: [batch_size, num_nodes, num_nodes] (32, 50, 50)

        batch_size, num_nodes = h.size(0), h.size(1)

        # Linear transformation
        Wh = self.W(h)  # [32, 50, n_heads*out_features] (32,50,4*32=128)
        Wh = Wh.view(batch_size, num_nodes, self.n_heads, self.out_features)  # [32,50,4,32]
        Wh = Wh.permute(0, 2, 1, 3)  # [32,4,50,32]

        # Compute attention coefficients
        Wh_repeated_i = Wh.unsqueeze(3).expand(-1, -1, -1, num_nodes, -1)  # [32,4,50,50,32]
        Wh_repeated_j = Wh.unsqueeze(2).expand(-1, -1, num_nodes, -1, -1)  # [32,4,50,50,32]
        concat = torch.cat([Wh_repeated_i, Wh_repeated_j], dim=-1)  # [32,4,50,50,64]

        e = self.leakyrelu(self.a(concat).squeeze(-1))  # [32,4,50,50]

        # Mask attention coefficients
        zero_vec = -9e15 * torch.ones_like(e)
        adj = adj.unsqueeze(1)  # [32,1,50,50]
        attention = torch.where(adj > 0, e, zero_vec)
        attention = F.softmax(attention, dim=-1)  # [32,4,50,50]
        attention = self.dropout_layer(attention)

        # Apply attention
        h_prime = torch.matmul(attention, Wh)  # [32,4,50,32]
        h_prime = h_prime.permute(0, 2, 1, 3).contiguous()  # [32,50,4,32]
        h_prime = h_prime.view(batch_size, num_nodes, -1)  # [32,50,128]

        return h_prime

#split out
class DualOutputGRAN(nn.Module):
    """
    Graph Recurrent Attention Network that generates both adjacency matrices and amino acid sequences.
    """
    def __init__(self, node_features=22, hidden_dim=128, num_layers=2,
                 n_heads=4, dropout=0.1, amino_acid_vocab_size=22):
        super(DualOutputGRAN, self).__init__()
        self.node_features = node_features
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.n_heads = n_heads
        self.amino_acid_vocab_size = amino_acid_vocab_size

        assert hidden_dim % n_heads == 0, "hidden_dim must be divisible by n_heads"
        self.out_features_per_head = hidden_dim // n_heads

        # Node feature embedding
        self.node_embedding = nn.Linear(node_features, hidden_dim)

        # Graph attention layers
        self.gat_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.gat_layers.append(GraphAttention(
                in_features=hidden_dim,
                out_features=self.out_features_per_head,
                n_heads=n_heads,
                dropout=dropout
            ))

        # Sequence generation
        self.rnn_cell = nn.GRUCell(hidden_dim, hidden_dim)
        self.sequence_projection = nn.Linear(hidden_dim, amino_acid_vocab_size)

        # Adjacency matrix generation
        self.edge_predictor = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

        self.dropout = nn.Dropout(dropout)

    def _process_graph(self, node_features, adjacency_matrix):
        """Process the input graph with graph attention layers"""
        h = self.node_embedding(node_features)  # [32,50,22] -> [32,50,128]

        for gat_layer in self.gat_layers:
            h = gat_layer(h, adjacency_matrix)  # [32,50,128]
            h = F.elu(h)
            h = self.dropout(h)

        return h

    def _generate_adjacency(self, node_embeddings):
        """Generate adjacency matrix from node embeddings"""
        batch_size, num_nodes, _ = node_embeddings.size()

        # Create all pairwise combinations of node embeddings
        node_i = node_embeddings.unsqueeze(2).repeat(1, 1, num_nodes, 1)  # [B, N, N, H]
        node_j = node_embeddings.unsqueeze(1).repeat(1, num_nodes, 1, 1)  # [B, N, N, H]

        # Concatenate node pairs
        node_pairs = torch.cat([node_i, node_j], dim=-1)  # [B, N, N, 2H]

        # Reshape for passing through edge predictor
        flat_pairs = node_pairs.view(-1, 2 * self.hidden_dim)  # [B*N*N, 2H]

        # Predict edges
        edge_scores = self.edge_predictor(flat_pairs).view(batch_size, num_nodes, num_nodes)  # [B, N, N]

        # Ensure symmetry (for undirected graphs)
        edge_scores = (edge_scores + edge_scores.transpose(1, 2)) / 2

        return edge_scores

    def forward(self, node_features, adjacency_matrix, target_sequences=None, target_adjacency=None, max_length=max_length):
        """
        Forward pass with dual outputs (sequence and adjacency matrix)

        Args:
            node_features: [batch_size, num_nodes, node_feature_dim]
            adjacency_matrix: [batch_size, num_nodes, num_nodes]
            target_sequences: [batch_size, seq_length] (for training)
            target_adjacency: [batch_size, num_nodes, num_nodes] (for training)
            max_length: Maximum sequence length for generation

        Returns:
            Dictionary containing:
                - 'sequence_logits': Predicted sequence logits
                - 'adjacency_matrix': Predicted adjacency matrix
                - Or generated sequence and adjacency matrix during inference
        """
        batch_size, num_nodes = node_features.size(0), node_features.size(1)

        # Process graph
        node_embeddings = self._process_graph(node_features, adjacency_matrix)  # [B, N, H]

        # Generate adjacency matrix
        predicted_adjacency = self._generate_adjacency(node_embeddings)  # [B, N, N]

        # Initialize RNN for sequence generation
        graph_embedding = torch.mean(node_embeddings, dim=1)  # [B, H]
        h_t = graph_embedding  # Initial hidden state

        if target_sequences is not None:
            # Training mode
            seq_length = target_sequences.size(1)
            sequence_logits = torch.zeros(batch_size, seq_length, self.amino_acid_vocab_size,
                                          device=node_features.device)

            x_t = torch.zeros(batch_size, self.hidden_dim, device=node_features.device)

            for t in range(seq_length):
                h_t = self.rnn_cell(x_t, h_t)  # [B, H]
                sequence_logits[:, t, :] = self.sequence_projection(h_t)  # [B, vocab_size]

                if t < seq_length - 1:
                    x_t = self.node_embedding(
                        F.one_hot(target_sequences[:, t],
                                  num_classes=self.amino_acid_vocab_size).float()
                    )  # [B, H]

            return {
                'sequence_logits': sequence_logits,
                'adjacency_matrix': predicted_adjacency
            }

        else:
            # Generation mode
            generated_sequences = torch.zeros(batch_size, max_length,
                                              dtype=torch.long,
                                              device=node_features.device)
            x_t = torch.zeros(batch_size, self.hidden_dim,
                              device=node_features.device)

            for t in range(max_length):
                h_t = self.rnn_cell(x_t, h_t)
                output = self.sequence_projection(h_t)
                prob = F.softmax(output, dim=-1)
                next_aa = torch.multinomial(prob, 1).squeeze(-1)
                generated_sequences[:, t] = next_aa
                x_t = self.node_embedding(
                    F.one_hot(next_aa, num_classes=self.amino_acid_vocab_size).float()
                )

            return {
                'generated_sequence': generated_sequences,
                'adjacency_matrix': predicted_adjacency
            }

    def compute_loss(self, predictions, targets):
        """
        Compute combined loss for sequence and adjacency matrix predictions

        Args:
            predictions: Dictionary from forward pass
            targets: Dictionary with 'sequence' and 'adjacency_matrix'

        Returns:
            Combined loss
        """
        # Sequence loss (cross entropy)
        seq_logits = predictions['sequence_logits']
        target_seq = targets['sequence']

        batch_size, seq_len, vocab_size = seq_logits.size()
        seq_logits_flat = seq_logits.view(-1, vocab_size)
        target_seq_flat = target_seq.view(-1)

        sequence_loss = F.cross_entropy(seq_logits_flat, target_seq_flat)

        # Adjacency matrix loss (binary cross entropy)
        pred_adj = predictions['adjacency_matrix']
        target_adj = targets['adjacency_matrix']

        # Apply mask to only consider non-diagonal elements
        mask = 1 - torch.eye(pred_adj.size(1), device=pred_adj.device).unsqueeze(0)
        adjacency_loss = F.binary_cross_entropy(
            pred_adj * mask,
            target_adj * mask,
            reduction='sum'
        ) / (mask.sum() + 1e-8)  # Normalize by number of edges

        # Combined loss (can be weighted if needed)
        combined_loss = sequence_loss + adjacency_loss

        return {
            'combined_loss': combined_loss,
            'sequence_loss': sequence_loss,
            'adjacency_loss': adjacency_loss
        }


# Data Loading and preprocessing of graphs, dataloaders

In [5]:

def load_protein_graph_data(parent_folder, max_proteins=None, chunk_length=chunk_length):
    """
    Load protein NetworkX graph data with detailed diagnostics
    """
    random.seed(42)

    # List to store graphs and their associated amino acid sequences
    protein_graphs = []
    protein_sequences = []

    # Diagnostic counters
    loaded_proteins = 0
    failed_proteins = 0

    folders = [name for name in os.listdir(parent_folder)
               if os.path.isdir(os.path.join(parent_folder, name))]

    found_folders = len(folders)
    print(f"Found {found_folders} protein folders")

    # Load only max_proteins if specified
    if max_proteins:
        folders = folders[:max_proteins]
        print(f"Limited to {len(folders)} folders due to max_proteins setting")

    for folder in folders:
        # Look for graph files in the folder
        folder_path = os.path.join(parent_folder, folder)
        graph_files = [f for f in os.listdir(folder_path) if f.endswith('_graph.pkl')]

        if graph_files:
            graph_file = os.path.join(folder_path, graph_files[0])
            try:
                with open(graph_file, 'rb') as f:
                    graph = pkl.load(f)

                    # Extract amino acid sequence
                    aa_seq = []
                    sorted_nodes = sorted(graph.nodes(), key=lambda x:
                    int(graph.nodes[x]['residue_number'])
                    if 'residue_number' in graph.nodes[x] else 0)

                    for node in sorted_nodes:
                        if 'residue_name' in graph.nodes[node]:
                            aa_seq.append(graph.nodes[node]['residue_name'])
                        else:
                            aa_seq.append('X')

                    protein_graphs.append(graph)
                    protein_sequences.append(aa_seq)
                    loaded_proteins += 1
            except Exception as e:
                failed_proteins += 1
                print(f"Error loading {graph_file}: {e}")

    print(f"Successfully loaded {loaded_proteins} protein graphs, {failed_proteins} failed")

    # Create smaller subgraphs
    subgraphs = []
    subsequences = []

    # Diagnostic counters for subsequence generation
    total_possible_subsequences = 0
    actual_generated_subsequences = 0
    proteins_with_no_subsequences = 0

    for i, (graph, sequence) in enumerate(zip(protein_graphs, protein_sequences)):
        # Extract nodes by residue number ranges
        sorted_nodes = sorted(graph.nodes(), key=lambda x:
        int(graph.nodes[x]['residue_number'])
        if 'residue_number' in graph.nodes[x] else 0)

        # Count theoretical maximum for this protein
        seq_length = len(sorted_nodes)
        max_subseqs_this_protein = seq_length  # With wrapping, we can generate as many subsequences as residues
        total_possible_subsequences += max_subseqs_this_protein

        subseqs_this_protein = 0

        # Create overlapping subsequences with wrapping
        for start_idx in range(0, seq_length, step_size):
            # Generate indices for the subsequence with wrapping
            indices = []
            for j in range(chunk_length):
                # Wrap around if we exceed the length
                wrapped_idx = (start_idx + j) % seq_length
                indices.append(wrapped_idx)

            # Get the corresponding nodes
            node_subset = [sorted_nodes[idx] for idx in indices]
            # Get the corresponding sequence
            aa_subset = [sequence[idx] for idx in indices]

            if node_subset:
                try:
                    subgraph = graph.subgraph(node_subset)
                    if len(subgraph) > 0:
                        subgraphs.append(subgraph)
                        subsequences.append(aa_subset)
                        subseqs_this_protein += 1
                        actual_generated_subsequences += 1
                except Exception as e:
                    print(f"Error creating subgraph: {e}")

        if subseqs_this_protein == 0:
            proteins_with_no_subsequences += 1

    print(f"\nSubsequence Generation Statistics:")
    print(f"Total possible subsequences: {total_possible_subsequences}")
    print(f"Actually generated subsequences: {actual_generated_subsequences}")
    print(f"Proteins with no subsequences: {proteins_with_no_subsequences}")

    # If no subgraphs were created, use the full graphs
    if not subgraphs:
        print("Warning: Could not create subgraphs. Using full graphs instead.")
        subgraphs = protein_graphs
        subsequences = protein_sequences

    print(f"Final count: {len(subgraphs)} protein subgraphs for training")

    # Shuffle the data
    if len(subgraphs) > 0:
        combined = list(zip(subgraphs, subsequences))
        random.shuffle(combined)
        subgraphs, subsequences = zip(*combined)

    # Limit size if needed
    if max_proteins and len(subgraphs) > subgraph_limit:
        print(f"Limiting from {len(subgraphs)} to {subgraph_limit} due to max_proteins /subgraph limit")
        subgraphs = subgraphs[:subgraph_limit]
        subsequences = subsequences[:subgraph_limit]

    return protein_graphs, protein_sequences, list(subgraphs), list(subsequences)

def prepare_graph_data_for_training(subgraphs, subsequences, unique_aa):
    """
    Prepare graph data for GRAN model training

    Args:
        subgraphs: List of NetworkX subgraphs
        subsequences: List of amino acid sequences corresponding to the subgraphs
        unique_aa: Set of unique amino acids to use for one-hot encoding

    Returns:
        aa_sequences_tensor: Tensor of amino acid sequences (targets)
        adjacency_tensors: Tensor of graph adjacency matrices
        node_features_tensor: Tensor of node features
    """
    # Prepare amino acid sequences (these will be our targets)
    aa_sequences = []
    for seq in subsequences:
        # Convert amino acid sequence to indices
        aa_indices = []
        for aa in seq:
            if aa in unique_aa:
                aa_indices.append(list(unique_aa).index(aa))
            else:
                # Handle unknown amino acids
                aa_indices.append(list(unique_aa).index('X') if 'X' in unique_aa else 0)
        aa_sequences.append(aa_indices)

    # Prepare adjacency matrices and node features
    adjacency_matrices = []
    node_features = []

    for i, graph in enumerate(subgraphs):
        # Get number of nodes
        num_nodes = len(graph)

        # Skip empty graphs
        if num_nodes == 0:
            continue

        # Create adjacency matrix
        adj_matrix = nx.to_numpy_array(graph)
        adjacency_matrices.append(adj_matrix)

        # Extract ordered nodes
        sorted_nodes = sorted(graph.nodes(), key=lambda x:
        int(graph.nodes[x]['residue_number'])
        if 'residue_number' in graph.nodes[x] else 0)

        # Create node features - use Meiler embedding if available
        features = np.zeros((num_nodes, len(unique_aa)))

        for j, node in enumerate(sorted_nodes):
            # Try to use meiler features if available (more informative)
            if 'meiler' in graph.nodes[node]:
                # Replace features with a better representation
                # For now, use one-hot as fallback
                aa = graph.nodes[node]['residue_name']
                if aa in unique_aa:
                    features[j, list(unique_aa).index(aa)] = 1.0
                elif 'X' in unique_aa:
                    features[j, list(unique_aa).index('X')] = 1.0
                else:
                    features[j, 0] = 1.0
            # Fallback to one-hot encoding
            elif 'residue_name' in graph.nodes[node]:
                aa = graph.nodes[node]['residue_name']
                if aa in unique_aa:
                    features[j, list(unique_aa).index(aa)] = 1.0
                elif 'X' in unique_aa:
                    features[j, list(unique_aa).index('X')] = 1.0
                else:
                    features[j, 0] = 1.0

        node_features.append(features)

    # Check if we have data
    if not adjacency_matrices or not node_features:
        raise ValueError("No valid graphs found after processing")

    # Convert to tensors
    aa_sequences_tensor = [torch.tensor(seq, dtype=torch.long) for seq in aa_sequences if seq]
    adjacency_tensors = [torch.tensor(adj, dtype=torch.float32) for adj in adjacency_matrices]
    node_features_tensor = [torch.tensor(nf, dtype=torch.float32) for nf in node_features]

    return aa_sequences_tensor, adjacency_tensors, node_features_tensor

def collate_batch(batch):
    """
    Custom collate function for padding sequences of variable length
    """
    # Handle potentially empty batches
    if not batch:
        return [], [], []

    aa_seqs, adjacency_matrices, node_feats = zip(*batch)

    # Handle potentially empty elements
    if not aa_seqs or not adjacency_matrices or not node_feats:
        return [], [], []

    # Pad sequences
    padded_aa_seqs = torch.nn.utils.rnn.pad_sequence(aa_seqs, batch_first=True)

    # Get maximum dimensions
    max_len = max([adj.size(0) for adj in adjacency_matrices])

    padded_adjacency_matrices = []
    padded_node_feats = []

    for i in range(len(adjacency_matrices)):
        adj = adjacency_matrices[i]
        nf = node_feats[i]

        pad_size = max_len - adj.size(0)
        if pad_size > 0:
            # Pad adjacency matrix
            padded_adj = F.pad(adj, (0, pad_size, 0, pad_size), "constant", 0)
            padded_adjacency_matrices.append(padded_adj)

            # Pad node features - make sure pad size isn't larger than twice the feature dim
            safe_pad_size = min(pad_size, nf.size(1) * 2)
            if safe_pad_size < pad_size:
                # We need to truncate the padding
                padded_nf = F.pad(nf, (0, 0, 0, safe_pad_size), "constant", 0)
                # Then add more nodes with zero features to match the adjacency matrix
                additional_pad = pad_size - safe_pad_size
                zeros = torch.zeros(additional_pad, nf.size(1))
                padded_nf = torch.cat([padded_nf, zeros], dim=0)
            else:
                padded_nf = F.pad(nf, (0, 0, 0, pad_size), "constant", 0)

            padded_node_feats.append(padded_nf)
        else:
            padded_adjacency_matrices.append(adj)
            padded_node_feats.append(nf)

    # Stack the padded tensors
    padded_adjacency_matrices = torch.stack(padded_adjacency_matrices)
    padded_node_feats = torch.stack(padded_node_feats)

    return padded_aa_seqs, padded_adjacency_matrices, padded_node_feats

def create_dataloader(aa_sequences, adjacency_matrices, node_features, batch_size=batch_size, shuffle=True):
    """
    Create a dataloader from the prepared data
    """
    dataset = list(zip(aa_sequences, adjacency_matrices, node_features))
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_batch
    )
    return dataloader



# Model loading if it already exists

In [6]:
def load_model(model_path, model, device):
    """
    Load a saved model - handles both formats

    Args:
        model_path: Path to the saved model
        model: Model instance to load the weights into
        device: Device to load the model to

    Returns:
        The loaded model
    """
    if not os.path.exists(model_path):
        print(f"No checkpoint found at {model_path}")
        return model

    try:
        # Try to load as a dictionary first
        checkpoint = torch.load(model_path, map_location=device)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        else:
            # Assume it's just the state dict
            model.load_state_dict(checkpoint)
        print(f"Successfully loaded model from {model_path}")
    except Exception as e:
        print(f"Error loading model: {e}")

    return model

# Load data -
some of this is just sanity checks and global things needed later, adjacency matrices, AA sequences

In [7]:

# Load protein graph data directly
try:
    full_graphs, full_sequences, subgraphs, subsequences = load_protein_graph_data(
        parent_folder, max_proteins, chunk_length
    )
except Exception as e:
    print(f"Error loading graph data: {e}")
    print("Current directory contains:", os.listdir())



# First graph debug
print("First few nodes of first graph:")
first_graph = full_graphs[0]
for i, node in enumerate(sorted(first_graph.nodes())[:5]):
    print(f"Node {node} attributes: {first_graph.nodes[node]}")

# Define a standard set of amino acids (all 20 standard ones)
STANDARD_AA = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
               'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL', 'X']

# Get unique amino acids from sequences but ensure we have at least the standard 20
UNIQUE_AA = set()
for seq in full_sequences:
    UNIQUE_AA.update(seq)

# Merge with standard AAs
UNIQUE_AA = sorted(list(set(UNIQUE_AA).union(set(STANDARD_AA))))
print(f"Unique amino acids: {len(UNIQUE_AA)}")
print(f"Amino acids found: {', '.join(UNIQUE_AA)}")

# Prepare data for training using graph data
aa_sequences, adjacency_matrices, node_features = prepare_graph_data_for_training(
    subgraphs, subsequences, UNIQUE_AA
)

print(f"Prepared data: {len(aa_sequences)} sequences, {len(adjacency_matrices)} adjacency matrices")


Found 3015 protein folders
Limited to 3000 folders due to max_proteins setting
Error loading nanos_networkx_small/5JDS_nanobody_B/5JDS_nanobody_B_graph.pkl: Ran out of input
Error loading nanos_networkx_small/8RW9_nanobody_C/8RW9_nanobody_C_graph.pkl: Ran out of input
Error loading nanos_networkx_small/8YBO_nanobody_B/8YBO_nanobody_B_graph.pkl: Ran out of input
Error loading nanos_networkx_small/6IBL_nanobody_C/6IBL_nanobody_C_graph.pkl: Ran out of input
Error loading nanos_networkx_small/8FQ7_nanobody_A/8FQ7_nanobody_A_graph.pkl: Ran out of input
Error loading nanos_networkx_small/2P43_nanobody_B/2P43_nanobody_B_graph.pkl: Ran out of input
Error loading nanos_networkx_small/7LVW_nanobody_I/7LVW_nanobody_I_graph.pkl: Ran out of input
Error loading nanos_networkx_small/7DSS_nanobody_A/7DSS_nanobody_A_graph.pkl: Ran out of input
Successfully loaded 2965 protein graphs, 8 failed

Subsequence Generation Statistics:
Total possible subsequences: 358689
Actually generated subsequences: 358689

In [12]:
import pickle as pkl
import numpy as np
import os
from datasets import Dataset, DatasetDict, Features, Value, Sequence
from huggingface_hub import HfApi, login
import pandas as pd
from tqdm import tqdm
import shutil
from datetime import datetime
import torch

def extract_gran_data(full_graphs, full_sequences, unique_aa):
    """Extract only essential data for GRAN - full proteins without subsequences"""

    # Create full proteins data
    full_proteins_data = []
    for i, (graph, seq) in enumerate(zip(full_graphs, full_sequences)):
        # Extract graph data
        adj_matrix = None
        if hasattr(graph, 'edges'):
            # Get adjacency matrix from graph
            import networkx as nx
            adj_matrix = nx.to_numpy_array(graph).tolist()

        # Extract meiler features if available
        node_features = []
        if hasattr(graph, 'nodes'):
            sorted_nodes = sorted(graph.nodes(), key=lambda x: int(graph.nodes[x].get('residue_number', 0)))
            for node in sorted_nodes:
                if 'meiler' in graph.nodes[node]:
                    # Convert pandas Series to list
                    meiler_data = graph.nodes[node]['meiler']
                    if hasattr(meiler_data, 'tolist'):
                        node_features.append(meiler_data.tolist())
                    else:
                        # In case it's already a list or array
                        node_features.append(list(meiler_data))
                else:
                    # Fallback to one-hot encoding if meiler features not available
                    aa = graph.nodes[node].get('residue_name', 'X')
                    aa_idx = unique_aa.index(aa) if aa in unique_aa else unique_aa.index('X') if 'X' in unique_aa else 0
                    one_hot = [0] * len(unique_aa)
                    one_hot[aa_idx] = 1
                    node_features.append(one_hot)

        protein_dict = {
            'sequence': seq,
            'sequence_length': len(seq),
            'graph_nodes': list(graph.nodes()) if hasattr(graph, 'nodes') else None,
            'graph_edges': list(graph.edges()) if hasattr(graph, 'edges') else None,
            'adjacency_matrix': adj_matrix,
            'node_features': node_features,
            'protein_id': f"protein_{i}"
        }
        full_proteins_data.append(protein_dict)

    # Create metadata
    metadata = {
        'unique_amino_acids': unique_aa,
        'num_proteins': len(full_proteins_data),
        'creation_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'average_protein_length': sum(len(seq) for seq in full_sequences) / len(full_sequences) if full_sequences else 0
    }

    return full_proteins_data, metadata

def upload_gran_dataset_to_hf(full_graphs, full_sequences, unique_aa,
                              dataset_name="gran-protein-structures",
                              username="alexchilton", save_local=True):
    """Upload essential GRAN protein dataset to Hugging Face account"""
    # Login to Hugging Face
    login()

    # Extract data
    print("Extracting GRAN data...")
    full_proteins_data, metadata = extract_gran_data(
        full_graphs, full_sequences, unique_aa
    )

    # Create dataset with explicit features
    print("Creating dataset...")

    # Define the features schema explicitly
    features = Features({
        'sequence': Sequence(Value('string')),
        'sequence_length': Value('int32'),
        'graph_nodes': Sequence(Value('string')),
        'graph_edges': Sequence(Sequence(Value('string'))),
        'adjacency_matrix': Sequence(Sequence(Value('float32'))),
        'node_features': Sequence(Sequence(Value('float32'))),
        'protein_id': Value('string')
    })

    protein_dataset = Dataset.from_pandas(pd.DataFrame(full_proteins_data), features=features)

    # Create a DatasetDict (single split for now)
    dataset_dict = DatasetDict({
        'train': protein_dataset
    })

    # Save locally before uploading if requested
    if save_local:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        local_dir = f"gran_protein_dataset_{timestamp}"
        print(f"Saving dataset locally to {local_dir}...")

        # Save using multiple formats for flexibility
        dataset_dict.save_to_disk(local_dir)

        # Also save raw data as pickle for backup
        raw_data = {
            'full_graphs': full_graphs,
            'full_sequences': full_sequences,
            'unique_aa': unique_aa,
            'metadata': metadata
        }
        with open(f"{local_dir}_raw_data.pkl", 'wb') as f:
            pkl.dump(raw_data, f)

        print(f"Local save complete. Files saved in {local_dir}")

    # Define a README.md content for the dataset card
    readme_content = f"""---
license: mit
task_categories:
- text-generation
- graph-ml
tags:
- protein
- graph-neural-network
- adjacency-matrix
- protein-structure
- nanobody
---

# GRAN Protein Structure Dataset

## Dataset Description

This dataset contains protein graph data for training Graph Recurrent Attention Networks (GRAN) for protein sequence and structure generation.

### Dataset Summary

- **Number of proteins:** {len(full_proteins_data)}
- **Average protein length:** {metadata['average_protein_length']:.1f} residues
- **Unique amino acids:** {len(metadata['unique_amino_acids'])}
- **Source:** Nanobody protein structures
- **Created by:** {username}
- **Date:** {metadata['creation_date']}

### Dataset Structure

Each protein entry contains:
- `sequence`: Complete amino acid sequence
- `sequence_length`: Total length of the protein
- `graph_nodes`: List of graph nodes (residue indices)
- `graph_edges`: List of graph edges (connections between residues)
- `adjacency_matrix`: Binary adjacency matrix representing contacts
- `node_features`: Features for each node (Meiler features or one-hot encoded residues)
- `protein_id`: Unique identifier

### Amino Acids

Available amino acids: {', '.join(metadata['unique_amino_acids'])}

### Usage

```python
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("{username}/{dataset_name}")

# Access a protein
protein = dataset['train'][0]
print(f"Sequence length: {{protein['sequence_length']}}")
print(f"Number of graph nodes: {{len(protein['graph_nodes'])}}")
print(f"Adjacency matrix shape: {{np.array(protein['adjacency_matrix']).shape}}")
```

### Training GRAN Model

This dataset is designed for training GRAN models that:
1. Generate both protein sequences and contact adjacency matrices
2. Model proteins as graphs with nodes (residues) and edges (contacts)
3. Use node features (Meiler descriptors or one-hot encoding)

### Citation

If you use this dataset, please cite:
```
@dataset{{gran_protein_structures,
  title={{GRAN Protein Structure Dataset}},
  author={{Alex Chilton}},
  year={{2025}},
  url={{https://huggingface.co/datasets/{username}/{dataset_name}}}
}}
```
"""

    # Upload to Hugging Face
    print(f"Uploading to {username}/{dataset_name}...")
    dataset_dict.push_to_hub(
        f"{username}/{dataset_name}",
        private=False,
        commit_message="Initial upload of GRAN protein structure dataset"
    )

    # Create and upload the README.md
    api = HfApi()
    api.upload_file(
        path_or_fileobj=readme_content.encode(),
        path_in_repo="README.md",
        repo_id=f"{username}/{dataset_name}",
        repo_type="dataset",
        commit_message="Add dataset card"
    )

    print(f"Successfully uploaded to https://huggingface.co/datasets/{username}/{dataset_name}")

    return dataset_dict

# Example usage - replace with your actual variables from GRAN notebook
# dataset = upload_gran_dataset_to_hf(
#     full_graphs=full_graphs,
#     full_sequences=full_sequences,
#     unique_aa=UNIQUE_AA,
#     dataset_name="gran-nanobody-proteins",
#     username="alexchilton",
#     save_local=True
# )

# To load the dataset later
# from datasets import load_dataset
# downloaded_dataset = load_dataset("alexchilton/gran-nanobody-proteins")
# print(f"Downloaded dataset contains {len(downloaded_dataset['train'])} proteins")

In [13]:
dataset = upload_gran_dataset_to_hf(
    full_graphs=full_graphs,
    full_sequences=full_sequences,
    unique_aa=UNIQUE_AA,
    dataset_name="gran-nanobody-proteins",
    username="alexchilton",
    save_local=True
)

Extracting GRAN data...
Creating dataset...
Saving dataset locally to gran_protein_dataset_20250504_160139...


Saving the dataset (0/1 shards):   0%|          | 0/2965 [00:00<?, ? examples/s]

Local save complete. Files saved in gran_protein_dataset_20250504_160139
Uploading to alexchilton/gran-nanobody-proteins...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Successfully uploaded to https://huggingface.co/datasets/alexchilton/gran-nanobody-proteins
